# Requirements
    - Pipeline
    - Normalize
    - Cross Validation
    - ML Regression
    - Evaluation Metric
    - LSTM
    - Evaluation Metric
        - Basic of Neural Networks
        - !batch size, !epoche, !learning rate, !learning rate dk

In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

# Load data
de_data = pd.read_csv("smard_inspection.txt", sep="\t")  # Replace with correct format if needed
uk_data = pd.read_csv("eso_inspection.txt", sep="\t")

# Preprocess Data
# Align datasets on time and relevant features
de_data['ID'] = pd.to_datetime(de_data['ID'])
uk_data['ID'] = pd.to_datetime(uk_data['ID'])
data = pd.merge(de_data, uk_data, on='ID', suffixes=('_DE', '_UK'))

# Select relevant features (adjust based on analysis)
features = [
    'Stromverbrauch_Gesamt (Netzlast) [MWh]',
    'Erzeugung_Wind Onshore [MWh]',
    'Erzeugung_Wind Offshore [MWh]',
    'Erzeugung_Photovoltaik [MWh]',
    'Stromfluss_Frankreich (Export) [MWh]',
    'Stromfluss_Frankreich (Import) [MWh]',
    'ENGLAND_WALES_DEMAND',
    'WIND',
    'SOLAR',
]
targets = [
    'Stromverbrauch_Gesamt (Netzlast) [MWh]',
    'ENGLAND_WALES_DEMAND',
    'Stromfluss_Nettoexport [MWh]',
]

data = data[features + targets].fillna(0)
X = data[features].values
y = data[targets].values

# Normalize data
from sklearn.preprocessing import MinMaxScaler
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)

# Create datasets
class TimeSeriesDataset(Dataset):
    def _init_(self, X, y, sequence_length):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.sequence_length = sequence_length

    def _len_(self):
        return len(self.X) - self.sequence_length

    def _getitem_(self, index):
        X_seq = self.X[index:index + self.sequence_length]
        y_seq = self.y[index + self.sequence_length - 1]
        return X_seq, y_seq

sequence_length = 24  # e.g., use 24 hours of data
dataset = TimeSeriesDataset(X_scaled, y_scaled, sequence_length)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

# Define LSTM Model
class LSTMModel(nn.Module):
    def _init_(self, input_size, hidden_size, output_size, num_layers=1):
        super(LSTMModel, self)._init_()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        h_n = h_n[-1]  # Take the last layer's output
        out = self.fc(h_n)
        return out

input_size = len(features)
hidden_size = 128
output_size = len(targets)
model = LSTMModel(input_size, hidden_size, output_size)

# Define training loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def train_model(model, train_loader, test_loader, epochs=10):
    model.train()
    for epoch in range(epochs):
        train_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {train_loss / len(train_loader)}")

    # Evaluate on test set
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            test_loss += loss.item()

    print(f"Test Loss: {test_loss / len(test_loader)}")

# Train the model
train_model(model, train_loader, test_loader, epochs=20)



ModuleNotFoundError: No module named 'torch'